# 01 — Exploratory Data Analysis

This notebook validates the Kaggle ULB dataset and examines class imbalance, transaction amount and time patterns.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from src.scorecard import load_creditcard_csv

DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'creditcard.csv'
FIGURE_DIR = PROJECT_ROOT / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')

In [ ]:
df = load_creditcard_csv(DATA_PATH)
print('Shape:', df.shape)
print('Duplicates:', df.duplicated().sum())
print('Missing values:', int(df.isna().sum().sum()))
df.head()

In [ ]:
class_summary = df['Class'].value_counts().rename_axis('Class').to_frame('Transactions')
class_summary['Percentage'] = class_summary['Transactions'] / len(df) * 100
class_summary

In [ ]:
ax = sns.countplot(data=df, x='Class', hue='Class', legend=False, palette=['#4C78A8', '#E45756'])
ax.set(title='Transaction Class Distribution', xlabel='Class (0=Normal, 1=Fraud)', ylabel='Transactions')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'fraud_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
plot_data = df.assign(Log_Amount=(df['Amount'] + 1).apply(__import__('numpy').log1p))
ax = sns.histplot(data=plot_data, x='Log_Amount', hue='Class', bins=60, stat='density', common_norm=False)
ax.set(title='Transaction Amount Distribution', xlabel='log(Amount + 1)')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'amount_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
df.assign(Hour=(df['Time'] / 3600) % 24).groupby('Class')[['Amount', 'Hour']].describe().round(2)

## EDA conclusion

Fraud is extremely rare, so accuracy is not an appropriate primary metric. The modeling notebooks use stratified splitting, class weighting, ROC-AUC, KS and score-band fraud rates.